# Broyles EPD Data Processing

Processing script for the Broyles compiled concrete EPD dataset (Compiled_Concrete_EPD_Data_Version_4c_Final_Published.xlsx).
Filters to relevant fields, adds GWP per cubic yard conversions, identifies SCM types (fly ash, slag), removes low-strength records, and applies IQR-based outlier removal consistent with the EC3 pipeline.

### Imports

In [ ]:
import os
import pathlib
import pandas as pd
import numpy as np

### Define Outlier Removal Functions

In [ ]:
def remove_outliers(df, col_names):
    """
    Remove extreme outliers based on IQR method across the full dataset.
    """
    q1 = df[col_names].quantile(0.25)
    q3 = df[col_names].quantile(0.75)
    iqr = q3 - q1

    df = df[~((df[col_names] < (q1 - 1.5 * iqr)) | (df[col_names] > (q3 + 1.5 * iqr))).any(axis=1)]

    return df.copy()


def remove_outliers_per_bucket(df, gwp_col, bucket_col):
    """
    Remove outliers within each compressive strength bucket using IQR method.
    More precise than global IQR since each strength class has a different GWP distribution.
    """
    def iqr_filter(group):
        q1 = group[gwp_col].quantile(0.25)
        q3 = group[gwp_col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        return group[(group[gwp_col] >= lower) & (group[gwp_col] <= upper)]

    return df.groupby(bucket_col, group_keys=False).apply(iqr_filter).reset_index(drop=True)

### Load Data

In [ ]:
# Determine repo root (works whether CWD is the notebook dir or repo root)
cwd = pathlib.Path(os.getcwd())
repo_root = cwd.parent if cwd.name == '03_processing_scripts' else cwd

excel_path = repo_root / '01_raw_data' / 'epd_data_Broyles' / 'Compiled_Concrete_EPD_Data_Version_4c_Final_Published.xlsx'
output_path = repo_root / '02_processed_data' / 'broyles_epd_data_cleaned.csv'

keep_cols = [
    'Company',
    'Company Location - Street',
    'Company Location - City',
    'Company Location - State',
    'Company Location - Zip',
    'Plant',
    'Plant Location - Street',
    'Plant Location - City',
    'Plant Location - State',
    'Plant Location - Zip',
    'U.S. Region of Plant',
    'EPD Program Operator',
    'EPD Date of Issue',
    'EPD Valid Until Date',
    'Mixture Label',
    'Mixture Description',
    'Concrete Compressive Strength (psi)',
    'Concrete Curation Time',
    'Declared Unit',
    'Product Components',
    'A1-A3 Global Warming Potential (kg CO2-eq)',
    'A1 GWP',
    'A2 GWP',
    'A3 GWP',
]

df = pd.read_excel(excel_path, usecols=keep_cols)

df = df.rename(columns={'Mixture Label': 'Mix Label', 'Mixture Description': 'Mix Description'})

# Coerce GWP columns to numeric (some cells use '-' as a placeholder for missing values)
gwp_raw_cols = [
    'A1-A3 Global Warming Potential (kg CO2-eq)',
    'A1 GWP',
    'A2 GWP',
    'A3 GWP',
]
for col in gwp_raw_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"Loaded {len(df)} records")
print(f"Columns: {list(df.columns)}")

### Add GWP per Cubic Yard Columns

The declared unit in this dataset is cubic meters (m³). Convert to per cubic yard using the factor: 1 CY = 0.764555 m³, so multiply m³ values by 0.764555 to get per-CY values.

In [ ]:
M3_TO_CY = 0.764555  # cubic meters per cubic yard

gwp_cols = [
    'A1-A3 Global Warming Potential (kg CO2-eq)',
    'A1 GWP',
    'A2 GWP',
    'A3 GWP'
]

for col in gwp_cols:
    df[f'{col}_per_CY'] = (df[col] * M3_TO_CY).round(2)

print("Added per-CY GWP columns:")
for col in gwp_cols:
    print(f"  {col}_per_CY")

### Identify SCM Types (Fly Ash, Slag)

Parse the `Product Components` field to flag mixes containing fly ash or slag.

In [ ]:
def field_contains(text, keyword):
    if pd.isna(text):
        return False
    return keyword.lower() in str(text).lower()

df['contains_fly_ash'] = df['Product Components'].apply(lambda x: field_contains(x, 'fly ash'))
df['contains_slag'] = df['Product Components'].apply(lambda x: field_contains(x, 'slag'))

print(f"Contains fly ash: {df['contains_fly_ash'].sum()} records ({df['contains_fly_ash'].mean():.1%})")
print(f"Contains slag:    {df['contains_slag'].sum()} records ({df['contains_slag'].mean():.1%})")

### Filter: Minimum Compressive Strength

In [ ]:
before = len(df)
df = df[df['Concrete Compressive Strength (psi)'] >= 2000].copy()
print(f"Removed {before - len(df)} records with compressive strength < 2000 psi ({len(df)} remaining)")

### Remove Global Outliers (IQR on Full Dataset)

Remove records where `A1-A3 Global Warming Potential (kg CO2-eq)_per_CY` falls outside 1.5×IQR of the global distribution.

In [ ]:
# Drop rows with NaN in the primary GWP column before outlier removal
gwp_col = 'A1-A3 Global Warming Potential (kg CO2-eq)_per_CY'

before = len(df)
df = df.dropna(subset=[gwp_col]).copy()
print(f"Dropped {before - len(df)} rows with missing A1-A3 GWP value ({len(df)} remaining)")

before = len(df)
df = remove_outliers(df, [gwp_col])
print(f"Global IQR: removed {before - len(df)} outlier rows ({len(df)} remaining)")

### Remove Per-Bucket Outliers (IQR Within Each Strength Class)

Round compressive strength to the nearest 500 psi to create buckets, then apply IQR filtering within each bucket.

In [ ]:
# Drop rows with NaN compressive strength before bucketing
before = len(df)
df = df.dropna(subset=['Concrete Compressive Strength (psi)']).copy()
if before - len(df) > 0:
    print(f"Dropped {before - len(df)} rows with missing compressive strength")

# Round to nearest 500 psi for bucket assignment
df['Compressive_Strength_Bucket'] = (
    (df['Concrete Compressive Strength (psi)'] / 500).round() * 500
).astype(int)

bucket_col = 'Compressive_Strength_Bucket'
before = len(df)
df = remove_outliers_per_bucket(df, gwp_col, bucket_col)
print(f"Per-bucket IQR: removed {before - len(df)} rows ({len(df)} remaining)")

df = df.drop(columns=['Compressive_Strength_Bucket'])

### Summary

In [ ]:
print(f"Final record count: {len(df)}")
print(f"Final column count: {len(df.columns)}")
print("\nColumn list:")
for col in df.columns:
    print(f"  {col}")

df.head(3)

### Save Cleaned Data

In [ ]:
df.to_csv(output_path, index=False)
print(f"Saved {len(df)} records to {output_path}")